LAB 4

In [10]:
con.execute("""
    CREATE OR REPLACE VIEW raw_transactions AS
    SELECT *, strftime(timestamp::TIMESTAMP, '%Y')::INT AS year,
              strftime(timestamp::TIMESTAMP, '%m')::INT AS month
    FROM read_csv_auto('bigdata/raw/transactions/transactions_synthetic.csv')
""")
print(con.execute("SELECT DISTINCT year, month FROM raw_transactions ORDER BY 1,2").fetchdf())

    year  month
0   2023      1
1   2023      2
2   2023      3
3   2023      4
4   2023      5
5   2023      6
6   2023      7
7   2023      8
8   2023      9
9   2023     10
10  2023     11
11  2023     12
12  2024      1
13  2024      2
14  2024      3
15  2024      4
16  2024      5
17  2024      6
18  2024      7
19  2024      8
20  2024      9
21  2024     10
22  2024     11
23  2024     12


In [11]:
con.execute("""
    COPY raw_transactions TO 'bigdata/silver/transactions_particionado'
    (FORMAT PARQUET, PARTITION_BY (year, month), OVERWRITE_OR_IGNORE 1)
""")

In [12]:
import time
t0 = time.time()
con.execute("SELECT COUNT(*) FROM raw_transactions WHERE month = 1").fetchall()
print("sem partição:", time.time() - t0, "s")

t0 = time.time()
con.execute("""
    SELECT COUNT(*) FROM read_parquet('bigdata/silver/transactions_particionado/year=*/month=1/*.parquet')
""").fetchall()
print("com partição:", time.time() - t0, "s")

sem partição: 0.21206021308898926 s
com partição: 0.0023794174194335938 s
